In [15]:
import math as m
import numpy as np
import random 
import stable_baselines3
#import gym 
import gymnasium as gym
import torch
import torch.nn as nn
import torch.nn.functional as F
#import tensorflow as tf
import random 

from abc import ABC, abstractmethod



In [16]:
import torch
import torch.nn as nn

print(torch.__version__)

net = nn.Linear(4, 4)
optimizer = torch.optim.SGD(net.parameters(), lr=3e-4)

print("works")

2.13.0+cu130
works


In [17]:
# redefine box and env  as in box is singular isolated , part of a bigger formula , pattern , structure made by the bigger class .
#test the stuff here 

class CellBox :
    def __init__(
            self,
            name,
            xy:tuple[int,int],
            color:str, 
            symbol:str=None, 
            dirlist : list = None, 
            reward_fn = None
        ):

        # this part describes the box itself
        self.name=name
        self.coordinate=xy
        self.color=color
        self.reward=0
        
        self.reward_fn = reward_fn if reward_fn != None else self.default_rewardfn
        
        self.symbol= symbol
        #this one is about the surrounding of the box
        self.neighbours={
            "up":dirlist[0],
            "down":dirlist[1],
            "right":dirlist[2],
            "left":dirlist[3],
            "stand":self
        }

    def default_rewardfn(self):
        if self.symbol == "X":
            self.reward = -100
        elif self.symbol == "O":
            self.reward = 100
        else:
            self.reward = -10

        return self.reward

    def reward_(self):
        return  self.reward_fn()

    def next_state(self,action):
        if self.neighbours[action] is None:
            return self
        return self.neighbours[action] 
    
    # 🔥 KEY FIX: equality based on identity of state meaning
    def __eq__(self, other):
        return isinstance(other, CellBox) and self.name == other.name

    def __hash__(self):
        return hash(self.name)

In [ ]:
class Board(ABC):

    def __init__(self,
                 cells: list[CellBox],
                 geometry="box",
                 data=None):

        self.cells = cells
        self.geometry = geometry
        self.data = data

        self.form()

    def form(self):

        if self.geometry == "box":
            self.make_box(self.data)

        elif self.geometry == "pyramid":
            self.make_pyramid(self.data)

        elif self.geometry == "stack":
            self.make_stack(self.data)

        elif self.geometry == "h_line":
            self.make_line({"direction": "h"})

        elif self.geometry == "v_line":
            self.make_line({"direction": "v"})

        elif self.geometry == "custom":
            self.make_custom(self.data)

    # -------------------------------------------------------

    def __iter__(self):
        return iter(self.cells)

    def __len__(self):
        return len(self.cells)

    def snapshot(self):
        return self.cells.copy()

    # -------------------------------------------------------

    def make_stack(self, data):
        pass

    def make_box(self, data):

        shape = data["shape"]
        N, M = shape

        cells = self.cells

        for m in range(M):
            for n in range(N):

                p = n + m * N
                s = cells[p]

                s.neighbours = {
                    "up": None,
                    "down": None,
                    "right": None,
                    "left": None,
                    "stand": s
                }

                if n < N - 1:
                    s.neighbours["right"] = cells[p + 1]

                if n > 0:
                    s.neighbours["left"] = cells[p - 1]

                if m < M - 1:
                    s.neighbours["up"] = cells[p + N]

                if m > 0:
                    s.neighbours["down"] = cells[p - N]

    def make_pyramid(self, data):
        pass

    def make_custom(self, data): 
        
        opposite = {
            "right": "left",
            "left":  "right",
            "up":    "down",
            "down":  "up",
            }
        # Reset every cell first
        for cell in self.cells:

            cell.neighbours = {
                "up": None,
                "down": None,
                "left": None,
                "right": None,
                "stand": cell,
            }

        # Apply all user-defined connections
        for cell_a, direction, cell_b in data["connections"]:

            if direction not in opposite:
                raise ValueError(
                    f"Unknown direction '{direction}'. "
                    f"Allowed: {list(opposite.keys())}"
                )

            # Forward connection
            cell_a.neighbours[direction] = cell_b

            # Reverse connection
            cell_b.neighbours[opposite[direction]] = cell_a

    def make_line(self, data):

        direction = data["direction"]

        if direction == "h":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["right"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["left"] = self.cells[i]

        elif direction == "v":

            for i in range(len(self.cells) - 1):

                self.cells[i].neighbours["up"] = self.cells[i + 1]
                self.cells[i + 1].neighbours["down"] = self.cells[i]

In [19]:
# Env (organizing the puzzle of state into env )
class Env:

    @property
    def actions(self) -> list:
        """All actions available in this environment."""
        raise NotImplementedError

    @property
    def states(self) -> list:
        """All states in this environment."""
        raise NotImplementedError

    def reset(self):
        """Start a new episode. Return the initial state."""
        raise NotImplementedError

    def step(self, action):
        """
        Apply action to the current state.
        Return (next_state, reward, done).
        """
        raise NotImplementedError
    
# class Chessboard(Env):
#     def __init__(self,
#                  states: list[CellBox],
#                  actions:list[str],
#                  start: CellBox):
#         self._states = states
#         self._actions = actions
#         self.start = start
#         self.current = start # the current state

#     @property
#     def states(self):
#         return self._states
#     @property
#     def actions(self):
#         return self._actions
    
#     # methode

#     def reset(self):
#         self.current = self.start
#         return self.current
    
#     def step(self, action):
#         """
#         Apply action to the current state.
 
#         Calls next_state()  → your CellBox transition logic
#         Calls reward_()     → your CellBox reward logic
 
#         Returns:
#             next_state  — the CellBox we landed on
#             reward      — float reward for this transition
#             done        — True if episode should end
#         """

#         next_state = self.current.next_state(action)
#         next_state.reward_()
#         reward = next_state.reward
#         done = next_state.symbol in ("X","O")

#         self.current = next_state 

#         return {
#             "state":self.current, 
#             "action" : action, 
#             "reward" : reward, 
#             "next_state":next_state , 
#             "done":done
#             }

In [20]:
class Chessboard(Env):
    """
    RL environment built on top of a Board.

    Board
        -> geometry

    Environment
        -> transitions
        -> rewards
        -> terminal conditions
        -> current state
    """

    def __init__(self,
                 board: Board,
                 actions: list[str],
                 start: CellBox):

        self.board = board

        self._actions = actions

        self.start = start
        self.current = start

    # -------------------------------------------------------

    @property
    def states(self):
        return self.board.cells

    @property
    def actions(self):
        return self._actions

    # -------------------------------------------------------

    def reset(self):

        self.current = self.start
        return self.current

    # -------------------------------------------------------

    def step(self, action):

        next_state = self.current.next_state(action)

        reward = next_state.reward_()

        done = next_state.symbol in ("X", "O")

        self.current = next_state

        return {
            "state": self.current,
            "action": action,
            "reward": reward,
            "next_state": next_state,
            "done": done,
        }

    # -------------------------------------------------------

    def snapshot(self):

        return {
            "current": self.current,
            "start": self.start,
            "board": self.board.snapshot(),
        }

In [ ]:
#set up
s1 = CellBox("s1",(0,0),"white","X",[None,None,None,None])
s2 = CellBox("s2",(2,0),"white","O",[None,None,None,None])
s3 = CellBox("s3",(4,0),"white","X",[None,None,None,None])
s4 = CellBox("s4",(0,1),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"grey",None,[None,None,None,None])
s6 = CellBox("s6",(2,1),"white",None,[None,None,None,None])
s7 = CellBox("s7",(3,1),"grey",None,[None,None,None,None])
s8 = CellBox("s8",(4,1),"white",None,[None,None,None,None])

#s0 = CellBox("empty box")

#define
s1.neighbours = {"up": s4,   "down": None, "right": None, "left": None, "stand": s1}
s2.neighbours = {"up": s6,   "down": None, "right": None, "left": None, "stand": s2}
s3.neighbours = {"up": s8,   "down": None, "right": None, "left": None, "stand": s3}
s4.neighbours = {"up": None, "down": s1,   "right": s5,   "left": None, "stand": s4}
s5.neighbours = {"up": None, "down": None, "right": s6,   "left": s4,   "stand": s5}
s6.neighbours = {"up": None, "down": s2,   "right": s7,   "left": s5,   "stand": s6}
s7.neighbours = {"up": None, "down": None, "right": s8,   "left": s6,   "stand": s7}
s8.neighbours = {"up": None, "down": s3,   "right": None, "left": s7,   "stand": s8}

#



States1=[s1,s2,s3,s4,s5,s6,s7,s8]

Actions=["up", "right","down","left"]




In [28]:
s1 = CellBox("s1",(2,0),"white",None,[None,None,None,None])
s2 = CellBox("s2",(2,1),"white",None,[None,None,None,None])
s3 = CellBox("s3",(2,2),"white",None,[None,None,None,None])
s4 = CellBox("s4",(1,0),"white",None,[None,None,None,None])
s5 = CellBox("s5",(1,1),"white","X",[None,None,None,None])
s6 = CellBox("s6",(1,2),"white",None,[None,None,None,None])
s7 = CellBox("s7",(0,0),"white",None,[None,None,None,None])
s8 = CellBox("s8",(0,1),"white",None,[None,None,None,None])
s9 = CellBox("s9",(0,2),"white","O",[None,None,None,None])

States2 = [s1,s2,s3,s4,s5,s6,s7,s8,s9]

claude_board = Board(States2,geometry="box",data={"shape":(3,3)})

env = Chessboard(
    board=claude_board,
    actions=Actions,
    start=s1
)

state = env.reset()

experience = env.step("left")

In [29]:
experience["next_state"].name

's1'

In [22]:
#reorganize the whole code here

def manhattan_goal(state:CellBox):
        x,y = state.coordinate
        return abs(x-0) + abs(y-2)

def manhattan_trap(state:CellBox):
        x,y = state.coordinate
        return abs(x-1) + abs(y-1)


#the state _ vector
def state_to_vector(state):

    color = 0 if state.color == "white" else 1

    if state.symbol == "X":
        symbol = -1
    elif state.symbol == "O":
        symbol = 1
    else:
        symbol = 0

    return np.array([
        state.coordinate[0],
        state.coordinate[1],
        color,
        symbol,
        manhattan_goal(state=state),
        manhattan_trap(state=state)
    ], dtype=np.float32)

In [23]:
# import the sb3 here and see how to integrate them with my code (see if i can package it under my designed classes)


In [24]:
# import the gymnasium stuff here